# 01 Filtriranje i izdvajanje tehnologija

Notebook radi dva koraka i upisuje jedan fajl:

1. iz 123.849 oglasa izdvaja IT oglase
2. iz njihovih opisa jezičkim modelom izvlači tehnologije i ostale oznake

Izlaz je `data/interim/dataset.parquet`, koji čita `02_analiza.ipynb`. Sadržaj
odgovara poglavljima 3 i 4 rada.

Pozivi modela se keširaju, pa ponovno pokretanje ne košta ništa.

## 1. Podešavanje

In [1]:
from pathlib import Path
import re
import pandas as pd

RAW     = Path("data/raw")
INTERIM = Path("data/interim")

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

## 2. Učitavanje

Koriste se tri tabele:

| tabela | sadržaj |
|---|---|
| `postings.csv` | jedan red je jedan oglas: naslov, opis, pregledi, prijave |
| `job_skills.csv` | LinkedIn oznaka funkcije posla (IT, SALE, ENG i slično) |
| `salaries.csv` | plate, u zasebnoj tabeli jer ih nema svaki oglas |

Iz `postings.csv` se čitaju samo potrebne kolone, jer fajl ima 493 MB.

In [2]:
postings = pd.read_csv(
    RAW / "postings.csv",
    usecols=[
        "job_id", "title", "description", "company_name",
        "views", "applies", "formatted_experience_level",
        "remote_allowed", "location", "formatted_work_type",
    ],
)
job_skills = pd.read_csv(RAW / "job_skills.csv")
salaries   = pd.read_csv(RAW / "salaries.csv")

print(f"postings   : {len(postings):>7,} oglasa")
print(f"job_skills : {len(job_skills):>7,} oznaka  ({job_skills['job_id'].nunique():,} oglasa)")
print(f"salaries   : {len(salaries):>7,} plata   ({salaries['job_id'].nunique():,} oglasa)")

postings   : 123,849 oglasa
job_skills : 213,768 oznaka  (126,807 oglasa)
salaries   :  40,785 plata   (40,785 oglasa)


### Provere ispravnosti

Uz korake stoje `assert` linije. Ako pretpostavka prestane da važi, na primer
posle nove verzije skupa podataka, notebook staje na tom mestu umesto da
proizvede pogrešan rezultat.

In [3]:
assert postings["job_id"].is_unique, "postings.csv ima duplirane job_id"
assert salaries["job_id"].is_unique, "salaries.csv ima duplirane job_id — merge bi duplirao oglase"

print("Pokrivenost kolona koje ćemo koristiti:")
for col in ["description", "views", "applies", "formatted_experience_level"]:
    n = postings[col].notna().sum()
    print(f"   {col:28} {n:>7,}  ({n/len(postings)*100:5.1f}%)")

Pokrivenost kolona koje ćemo koristiti:
   description                  123,842  (100.0%)
   views                        122,160  ( 98.6%)
   applies                       23,320  ( 18.8%)
   formatted_experience_level    94,440  ( 76.3%)


## 3. Filter, prvi sloj: LinkedIn oznaka funkcije posla

Filtriranje po delatnosti firme ne razlikuje poslove unutar iste firme. Meta je
u kategoriji Technology i za backend inženjera i za office managera.

`job_skills.csv` sadrži oznaku funkcije posla na nivou oglasa, pa oznaka IT
znači da je LinkedIn sam posao svrstao u IT. Jedan oglas može imati više
oznaka, zato se radi sa skupom job_id vrednosti; spajanje tabela bi udvostručilo
redove.

In [4]:
print(job_skills["skill_abr"].value_counts().head(10).to_string())

it_job_ids = set(job_skills.loc[job_skills["skill_abr"] == "IT", "job_id"])
print(f"\nOglasa sa oznakom IT: {len(it_job_ids):,}")

skill_abr
IT      26137
SALE    22475
MGMT    20861
MNFC    18185
HCPR    17369
BD      14290
ENG     13009
OTHR    12608
FIN      8540
MRKT     5525

Oglasa sa oznakom IT: 26,137


## 4. Filter, drugi sloj: naziv pozicije

LinkedIn oznaka IT je široka, a na nekim mestima i preuska, pa se uz nju gleda i
naziv pozicije. Koriste se tri liste:

| lista | uloga |
|---|---|
| STRONG | naziv koji može značiti samo IT posao |
| INCLUDE | naziv koji upućuje na IT, ali može biti i nešto drugo |
| EXCLUDE | naziv koji upućuje na posao izvan IT-a |

```
prolazi = (IT oznaka i (STRONG ili (INCLUDE i ne EXCLUDE))) ili STRONG
```

STRONG prolazi i bez IT oznake. LinkedIn oznaka opisuje funkciju posla, a
Engineering, Analyst, QA i Research su kod njih zasebne kategorije, pa naziv
Senior Software Engineer često nosi oznaku ENG. Takvih oglasa ima 1.525, a
odluku o njima donosi jezički model u trećem sloju.

STRONG prelazi i preko EXCLUDE liste, jer reči u njoj opisuju oblast, a ne
funkciju. Data Analyst, Marketing je analitičar podataka koji radi u marketingu.

Pojmovi quality assurance i test engineer su premešteni iz STRONG u širu listu,
jer pogađaju i kontrolu kvaliteta u farmaciji i ispitivanje hardvera.

Sve reči se traže sa granicom reči (`\b`). Bez toga se `ai` javlja u 8.068
naslova unutar drugih reči (Maintenance, Retail), a `hr` u 515 (Threat,
Chromium), pa bi takvi oglasi ulazili ili ispadali bez veze sa poslom.

In [5]:
# Nazivi koji ne mogu značiti ništa osim IT-a — prolaze bezuslovno.
STRONG = [
    "software engineer", "software developer", "software architect",
    "developer", "programmer",
    "data scientist", "data analyst", "data engineer", "data architect",
    "machine learning", "deep learning", "mlops", "data science",
    "devops", "devsecops", "sre", "site reliability", "platform engineer",
    "backend", "back-end", "frontend", "front-end",
    "full stack", "fullstack", "full-stack",
    "cloud engineer", "cloud architect", "solutions architect",
    "enterprise architect", "systems architect",
    "security engineer", "security analyst", "cybersecurity",
    "cyber security", "penetration tester", "infosec",
    "qa engineer", "sdet",
    "database administrator", "dba",
    "systems administrator", "system administrator", "sysadmin",
    "business intelligence", "it support", "help desk", "helpdesk",
]

# Previše višeznačni da bi gazili EXCLUDE, ali korisni kao širi pojam.
# "quality assurance" pogađa i farmaceutsku, prehrambenu i proizvodnu kontrolu
# kvaliteta, "test engineer" i električno i termičko ispitivanje hardvera.
DVOSMISLENI = ["quality assurance", "test engineer"]

# Sugerišu IT, ali mogu značiti i nešto drugo.
INCLUDE = STRONG + DVOSMISLENI + [
    "software", "engineer", "architect", "analyst", "scientist", "analytics",
    "data", "web", "mobile", "ios", "android",
    "cloud", "platform", "network", "database",
    "security", "cyber", "qa", "tester",
    "ml", "ai", "nlp", "it", "firmware", "embedded",
]

# Sugerišu da posao nije IT.  Gube od STRONG liste.
EXCLUDE = [
    # prodaja i administracija
    "sales", "telesales", "marketing", "recruiter", "recruiting", "talent",
    "account executive", "office manager", "clerk", "data entry",
    "hr", "hris", "human resources", "legal", "payroll",
    # zdravstvo i nauka van IT-a
    "clinical", "laboratory", "biomarker", "pharmacy", "pharmaceutical",
    "nurse", "chemist", "biolog",
    # ostale inženjerske struke
    "civil", "highway", "structural", "geotechnical", "mechanical",
    "electrical", "chemical", "automotive", "aerospace", "thermal",
    "marine", "maritime", "surface mount",
    # zanati i održavanje
    "hvac", "plumbing", "electrician", "welder", "welding",
    "mechanic", "maintenance", "technician trainee", "driver",
    # obrazovanje
    "teacher", "professor", "instructor",
]


def _rx(words: list[str]) -> str:
    """Regex koji traži bilo koju reč iz liste, ali samo kao celu reč (\b)."""
    return r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b"


title = postings["title"].astype(str).str.lower()

has_strong  = title.str.contains(_rx(STRONG),  regex=True, na=False)
has_include = title.str.contains(_rx(INCLUDE), regex=True, na=False)
has_exclude = title.str.contains(_rx(EXCLUDE), regex=True, na=False)
has_it_tag  = postings["job_id"].isin(it_job_ids)

# LinkedIn-ova oznaka posla je taksonomija FUNKCIJE, u kojoj su Engineering,
# Analyst, QA i Research zasebne kategorije od IT-a.  Zbog toga "Senior Software
# Engineer" često nosi oznaku ENG umesto IT.  Nedvosmislen naziv (STRONG) zato
# prolazi i bez oznake; konačnu odluku svejedno donosi LLM u sloju 3.
prolazi_naslov = has_strong | (has_include & ~has_exclude)
postings["prolazi_filter"] = (has_it_tag & prolazi_naslov) | has_strong

print(f"Prolazi naslov               : {prolazi_naslov.sum():,}")
print(f"Naslov + IT oznaka           : {(has_it_tag & prolazi_naslov).sum():,}")
print(f"Dodato: STRONG bez IT oznake : {(has_strong & ~(has_it_tag & prolazi_naslov)).sum():,}")
print(f"Ukupno prolazi filter        : {postings['prolazi_filter'].sum():,}")
print(f"Spaseno STRONG listom        : {(has_it_tag & has_strong & has_exclude).sum():,}")

Prolazi naslov               : 21,913
Naslov + IT oznaka           : 13,781
Dodato: STRONG bez IT oznake : 1,525
Ukupno prolazi filter        : 15,306
Spaseno STRONG listom        : 56


### Levak: koliko otpada na svakom koraku

Kolona sa platom prati koliko oglasa sa navedenom platom ostaje posle svakog
koraka, jer se na njima radi analiza plata.

In [6]:
sal_ids = set(salaries["job_id"])

koraci = [
    ("svi oglasi",                pd.Series(True, index=postings.index)),
    ("+ LinkedIn IT oznaka",      has_it_tag),
    ("+ naslov ima IT pojam",     has_it_tag & has_include),
    ("- naslov ima ne-IT pojam",  has_it_tag & has_include & ~has_exclude),
    ("+ vraćeni STRONG listom",   has_it_tag & prolazi_naslov),
    ("+ STRONG bez IT oznake",    postings["prolazi_filter"]),
]

redovi = []
for naziv, maska in koraci:
    ids = set(postings.loc[maska, "job_id"])
    redovi.append({"korak": naziv, "oglasa": len(ids), "sa platom": len(ids & sal_ids)})

levak = pd.DataFrame(redovi)
levak["% od prethodnog"] = (levak["oglasa"] / levak["oglasa"].shift(1) * 100).round(1)
print(levak.to_string(index=False))

                   korak  oglasa  sa platom  % od prethodnog
              svi oglasi  123849      36073              NaN
    + LinkedIn IT oznaka   25256       7177             20.4
   + naslov ima IT pojam   14710       4208             58.2
- naslov ima ne-IT pojam   13725       3955             93.3
 + vraćeni STRONG listom   13781       3975            100.4
  + STRONG bez IT oznake   15306       4404            111.1


### Kontrola uzorka

Pregled zadržanih i izbačenih naslova pokazuje da li su liste preuske ili
preširoke.

In [7]:
prosli   = postings.loc[postings["prolazi_filter"], "title"]
spaseni  = postings.loc[has_it_tag & has_strong & has_exclude, "title"]
izbaceni = postings.loc[has_it_tag & has_include & has_exclude & ~has_strong, "title"]

print(f"ZADRŽANI (uzorak od {len(prosli):,}):")
print(prosli.sample(15, random_state=1).to_string(index=False))

print(f"\n\nVRAĆENI STRONG listom — oblast nije IT, ali funkcija jeste ({len(spaseni):,}):")
print(spaseni.sample(10, random_state=1).to_string(index=False))

print(f"\n\nIZBAČENI ({len(izbaceni):,}):")
print(izbaceni.sample(15, random_state=1).to_string(index=False))

ZADRŽANI (uzorak od 15,306):
                    Power BI Business Analyst
           Vice President of Data Engineering
              Entry-Level Validation Engineer
                    Software Engineer in Test
.Net UI Developer(AZURE)/Cambridge, MA-Onsite
 Senior Programmer Analyst, Insight Analytics
                     Sr Dell Boomi Developer 
             Senior Site Reliability Engineer
                  Azure Synapse Data Engineer
                          AWS DevOps Engineer
                    Angular Software Engineer
           WORDPRESS E-COMMERCE Web Developer
                  Network Security Specialist
                       Network Engineer - III
                          .NET Build Engineer


VRAĆENI STRONG listom — oblast nije IT, ali funkcija jeste (56):
                ServiceNow Developer, Transaction Banking, Ops & Sales
                         Sr Advanced Aerospace Cyber Security Engineer
                               Data Science Manager Sales Ops (Remote)
 

## 5. Filter, treći sloj: šema za jezički model

Deo oglasa se po nazivu ne može razvrstati:

```
Transportation and Installation Engineer Offshore Wind   nije IT
Cloud Solutions Engineer                                 jeste IT
```

Razlika je u opisu posla, pa taj deo čita model.

### Oblik odgovora

Odgovor je opisan Pydantic šemom koja se šalje uz zahtev (structured output).
Groq proverava odgovor u odnosu na šemu posle generisanja, pa neusklađen odgovor
vraća grešku, a poziv se ponavlja.

### Redosled polja

Polje `reasoning` je prvo. Model generiše JSON redom, pa ono što napiše u tom
polju ulazi u kontekst dok popunjava ostala. Bez toga je obrazloženje završavalo
u listama:

```json
"red_flags": ["fast-paced" is not mentioned, but "high-quality functions"
              imply a demanding environment]
```

Postupak se naziva chain-of-thought via schema ordering.

### Polja

`is_it_role` je treći sloj filtera, a `role_family` gruba podela uloga, koja se
koristi u poglavlju 7 rada. U istom prolazu se izdvajaju i tehnologije, ton
oglasa i zastavice.

In [8]:
from typing import Literal
from pydantic import BaseModel, Field

# vibe_score MORA biti enum, ne int.  Strict mode garantuje tip, ali `int`
# nema gornju granicu — u testu je model vratio 70 na skali 1-10.
VibeScore = Literal[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

RoleFamily = Literal[
    "software_engineering",   # backend, frontend, full-stack, mobile, embedded software
    "data",                   # data engineer/analyst/scientist, BI, ML
    "devops_cloud",           # devops, SRE, platform, cloud, infrastruktura
    "security",               # cybersecurity, infosec, IAM
    "qa",                     # QA, SDET, test automation
    "it_support",             # helpdesk, sysadmin, desktop support, network admin
    "management",             # engineering manager, tech lead bez hands-on rada
    "non_it",                 # nije IT pozicija
]


class OglasAnaliza(BaseModel):
    """Struktura koju model mora da vrati za svaki oglas."""

    reasoning: str = Field(
        description="INTERNAL THOUGHTS ONLY. Analyze the posting here first. "
                    "Do NOT repeat this text in the lists below."
    )
    is_it_role: bool = Field(
        description="True only if the core work is building, running or securing "
                    "software, data or IT systems."
    )
    role_family: RoleFamily = Field(
        description="Primary function of the role. Use 'non_it' if is_it_role is false."
    )
    tech_stack: list[str] = Field(
        description="Specific tools, languages, frameworks, platforms. "
                    "NO SENTENCES. Example: ['Java', 'AWS', 'Docker']. "
                    "Empty list if none are named."
    )
    vibe_score: VibeScore = Field(
        description="Tone of the posting, 1 (chaotic startup) to 10 (ultra-corporate)."
    )
    red_flags: list[str] = Field(
        description="EXACT SHORT PHRASES (max 5 words) signalling poor conditions. "
                    "Empty list if none."
    )
    green_flags: list[str] = Field(
        description="EXACT SHORT PHRASES (max 5 words) naming tangible benefits. "
                    "Empty list if none."
    )


print("Polja u šemi:", list(OglasAnaliza.model_fields))

Polja u šemi: ['reasoning', 'is_it_role', 'role_family', 'tech_stack', 'vibe_score', 'red_flags', 'green_flags']


### Prompt

Prompt opisuje šta se smatra tehnologijom i kako se odlučuje da li je pozicija
IT. Odluku određuje funkcija posla, a ne delatnost firme: analitičar podataka u
marketingu jeste IT pozicija, elektroinženjer u softverskoj firmi nije. Za
red_flags, green_flags i vibe_score navedeni su i primeri onoga što se ne
računa.

In [9]:
SYSTEM_PROMPT = """You are a strict Data Extraction Robot acting as an expert HR Data Analyst.

CRITICAL RULES - VIOLATION = FAILURE:
1. NEVER write sentences or explanations inside the lists
2. NEVER write "X is not mentioned but Y..." - just extract Y or return []
3. Each list item MUST be 1-5 words maximum
4. If nothing is found, return an EMPTY LIST []
5. Use the 'reasoning' field to think. Everything else is pure extraction.

IS_IT_ROLE - decide by FUNCTION, not by industry:
TRUE:  software/data/QA/devops/security/IT-support work, database administration,
       system administration, technical architecture
FALSE: mechanical, civil, electrical, chemical or aerospace engineering;
       laboratory and clinical science; field technicians and maintenance;
       sales, marketing, HR, finance roles; construction project management
A data analyst working in marketing is TRUE (function is data).
An electrical engineer at a software company is FALSE (function is hardware).
If the posting is mostly about physical equipment, return FALSE.

TECH STACK = ONLY software, tools, languages, platforms:
YES: Python, AWS, React, Docker, SQL, Kubernetes, Salesforce
NO:  "Fortune 500 company", "HVAC", "strong communication skills",
     "Computer Science" (that is a degree, not a technology)

RED FLAGS = phrases indicating:
- Toxic culture: "we are a family", "wear many hats"
- Poor work-life balance: "fast-paced", "work under pressure", "weekend availability"
- Vague expectations: "self-starter", "hit the ground running"
NOT red flags: "on-site role" (logistics), "strong communication skills"
(requirement), "zero tolerance for discrimination" (policy)

GREEN FLAGS = ONLY tangible benefits:
"401k with company match", "unlimited PTO", "remote work option", "health insurance"
NOT green flags: "collaborative environment", "growth opportunities",
"challenging work" - these are marketing, not benefits

VIBE SCORE:
1-2  chaotic startup: "move fast break things", "equity over salary", "founding team"
3-4  casual startup: "rockstar", "ninja", "beer fridge", "choose your own schedule"
5-6  balanced: professional but friendly, standard benefits
7-8  corporate formal: "strategic alignment", "cross-functional", "stakeholder management"
9-10 ultra-corporate: "synergy", "governance", "compliance", shift work, strict hours
Defaults: "fast-paced" alone = 7. Shift work = 9. "startup" = 2-3.

ROLE_FAMILY - output EXACTLY ONE of these eight strings, nothing else:
software_engineering | data | devops_cloud | security | qa | it_support | management | non_it
Cloud, networking, infrastructure and platform roles -> devops_cloud
Helpdesk, desktop support, sysadmin -> it_support
Never invent a new value.

Keep 'reasoning' under 35 words. It is scratch space, not a report.
"""

USER_PROMPT = "Job title: {title}\n\nJob posting:\n{description}"

print(f"System prompt: {len(SYSTEM_PROMPT):,} karaktera "
      f"(~{len(SYSTEM_PROMPT)//4:,} tokena po pozivu)")

System prompt: 2,763 karaktera (~690 tokena po pozivu)


## 6. Keš i obrada

Ekstrakcija je jedini korak koji košta, pa se svaki oglas plaća jednom. Obrađen
oglas se odmah dopisuje u `data/interim/ekstrakcija_<model>.jsonl`, a pri
pokretanju se preskaču job_id vrednosti koje su već tamo. Prekid ćelije, pad
veze ili dodavanje novih oglasa zato ne znače ponovno plaćanje. Keš je odvojen
po modelu, pa se rezultati dva modela mogu porediti.

Tri podešavanja poziva:

`reasoning_effort="low"` jer gpt-oss modeli troše tokene na skriveno
razmišljanje pre odgovora, a zadatak je izdvajanje podataka. Mereno na pet
oglasa: 216 izlaznih tokena umesto 702.

`temperature=0` da isti oglas daje isti odgovor, što je uslov za merenja koja
slede.

`MAX_ZNAKOVA = 20_000`, praktično bez skraćivanja, jer najduži opis ima 10.004
znaka. Koliko je skraćivanje na 2.500 znakova menjalo rezultat izmereno je u
sekciji 8.

Svaki oglas se šalje zasebno, u više niti. Greška u jednom oglasu ne utiče na
ostale i ne upisuje se u keš, pa se taj oglas ponavlja pri sledećem pokretanju.

In [10]:
import json
import os
import threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import random
import time

from dotenv import load_dotenv
from openai import BadRequestError, OpenAI, RateLimitError
from tqdm.auto import tqdm

load_dotenv()

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",   # Groq je API-kompatibilan sa OpenAI
)

MAX_ZNAKOVA = 20_000   # zaštitna granica; najduži opis u datasetu ima 10.004 znaka

_lock = threading.Lock()


def putanja_kesa(model: str, oznaka: str | None = None) -> Path:
    prefiks = f"{oznaka}_" if oznaka else ""
    return INTERIM / f"ekstrakcija_{prefiks}{model.replace('/', '_')}.jsonl"


def ucitaj_kes(model: str, oznaka: str | None = None) -> dict[int, dict]:
    """Vraća {job_id: rezultat} iz keša na disku."""
    p = putanja_kesa(model, oznaka)
    if not p.exists():
        return {}
    with p.open(encoding="utf-8") as f:
        return {r["job_id"]: r for r in (json.loads(line) for line in f)}


def _obradi_jedan(red, model: str, granica: int | None = None) -> dict:
    poruke = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT.format(
            title=red.title, description=str(red.description)[:granica or MAX_ZNAKOVA])},
    ]
    # Ponavlja se zbog rate limita (429) i zbog odgovora koji ne odgovara šemi.
    # Uz temperature=0 isti zahtev daje istu grešku, pa drugi pokušaj podiže temperaturu.
    temp = 0                      # 0 = reproducibilnost
    for pokusaj in range(6):
        try:
            odgovor = client.chat.completions.parse(
                model=model,
                messages=poruke,
                response_format=OglasAnaliza,
                reasoning_effort="low",   # bez ovoga model troši 3x više izlaznih tokena
                temperature=temp,
            )
            break
        except RateLimitError:
            time.sleep(min(2 ** pokusaj, 30) + random.random() * 2)
        except BadRequestError:
            if temp == 0:
                temp = 0.6
            elif pokusaj >= 2:
                raise
    else:
        raise RuntimeError("rate limit posle 6 pokušaja")

    return {
        "job_id": int(red.job_id),
        **odgovor.choices[0].message.parsed.model_dump(),
        "_tokena_ulaz": odgovor.usage.prompt_tokens,
        "_tokena_izlaz": odgovor.usage.completion_tokens,
    }


def ekstrahuj(df: pd.DataFrame, model: str, niti: int = 6,
              oznaka: str | None = None, samo_kes: bool = False,
              granica: int | None = None) -> pd.DataFrame:
    """Obradi oglase iz df; preskače sve što je već u kešu.

    samo_kes=True  → ne poziva API, vraća samo ono što već postoji na disku.
    """
    kes = ucitaj_kes(model, oznaka)
    preostali = df[~df["job_id"].isin(kes)]

    if samo_kes:
        print(f"{model}: iz keša {len(kes):,} | neobrađeno {len(preostali):,} (API nije pozvan)")
    elif len(preostali):
        print(f"{model}: u kešu {len(kes):,} | za obradu {len(preostali):,}")
        p = putanja_kesa(model, oznaka)
        greske = Counter()
        with p.open("a", encoding="utf-8") as f, ThreadPoolExecutor(niti) as pool:
            poslovi = [pool.submit(_obradi_jedan, red, model, granica)
                       for red in preostali.itertuples()]
            for fut in tqdm(as_completed(poslovi), total=len(poslovi)):
                try:
                    rez = fut.result()
                except Exception as e:
                    greske[type(e).__name__] += 1
                    continue                      # ne kešira se → ponoviće se
                with _lock:
                    f.write(json.dumps(rez, ensure_ascii=False) + "\n")
                    f.flush()
                kes[rez["job_id"]] = rez
        if greske:
            print(f"neuspelo {sum(greske.values())}: {dict(greske)}")
            print("pokreni ćeliju ponovo da se ponove — uspešni se preskaču")
    else:
        print(f"{model}: sve {len(kes):,} već u kešu")

    return pd.DataFrame([kes[i] for i in df["job_id"] if i in kes])

## 7. Izbor modela

Oba kandidata obrađuju istih 200 oglasa, pa se meri koliko se slažu. Ako
jeftiniji model daje isti rezultat, nema razloga plaćati skuplji.

Slaganje na spisku tehnologija meri se Jaccardovim indeksom:

```
J(A, B) = |A ∩ B| / |A ∪ B|
```

Za `{Python, AWS, SQL}` i `{Python, AWS}` presek ima dva člana, unija tri, pa je
J = 0,67.

Za `is_it_role` koristi se Cohenova kapa, koja odbija slaganje koje bi nastalo
slučajno. Visoka vrednost znači da dva modela daju isti odgovor, ali ne dokazuje
da je odgovor tačan, jer su modeli iz iste porodice i dele greške. Tačnost meri
ručna provera u sekciji 12.

Pilot je pokrenut pre uvođenja punog opisa i temperature 0. Uslovi su bili isti
za oba modela, pa poređenje važi, a apsolutan broj tehnologija po oglasu je niži
nego u konačnoj ekstrakciji.

In [11]:
filtrirani = postings[postings["prolazi_filter"]].copy()

# Pilot je fiksiran na istih 200 oglasa na kojima je poređenje modela i izvedeno.
# Da se uzorkuje iznova (`filtrirani.sample(200, random_state=42)`), svaka kasnija
# izmena filtera promenila bi uzorak, pa izmereno slaganje modela više ne bi
# odgovaralo onome što je zapisano u tekstu.
_pilot_ids = set(ucitaj_kes("openai/gpt-oss-120b", oznaka="pilot"))
if _pilot_ids:
    pilot = filtrirani[filtrirani["job_id"].isin(_pilot_ids)].copy()
else:
    pilot = filtrirani.sample(200, random_state=42)

print(f"Filtrirani skup : {len(filtrirani):,} oglasa")
print(f"Pilot uzorak    : {len(pilot):,} oglasa")


Filtrirani skup : 15,306 oglasa
Pilot uzorak    : 200 oglasa


In [12]:
MODEL_JEFTIN = "openai/gpt-oss-20b"
MODEL_SKUP   = "openai/gpt-oss-120b"

# samo_kes=True → čita rezultate pilota sa diska umesto da ih plaća ponovo.
# Postavi na False samo ako želiš da ponoviš pilot.
rez_jeftin = ekstrahuj(pilot, MODEL_JEFTIN, oznaka="pilot", samo_kes=True)
rez_skup   = ekstrahuj(pilot, MODEL_SKUP,   oznaka="pilot", samo_kes=True)

openai/gpt-oss-20b: iz keša 194 | neobrađeno 6 (API nije pozvan)
openai/gpt-oss-120b: iz keša 200 | neobrađeno 0 (API nije pozvan)


### Stvarna potrošnja i cena

In [13]:
CENE = {   # $ po milionu tokena, Groq
    "openai/gpt-oss-20b":  (0.075, 0.30),
    "openai/gpt-oss-120b": (0.15,  0.60),
}

redovi = []
for model, rez in [(MODEL_JEFTIN, rez_jeftin), (MODEL_SKUP, rez_skup)]:
    c_in, c_out = CENE[model]
    ul, iz = rez["_tokena_ulaz"].mean(), rez["_tokena_izlaz"].mean()
    redovi.append({
        "model": model.split("/")[-1],
        "uspešno": len(rez),
        "neuspelo": len(pilot) - len(rez),
        "tokena ulaz": round(ul),
        "tokena izlaz": round(iz),
        "cena pilota $": round(len(rez) * (ul * c_in + iz * c_out) / 1e6, 3),
        "cena 15.306 $": round(len(filtrirani) * (ul * c_in + iz * c_out) / 1e6, 2),
    })

print(pd.DataFrame(redovi).to_string(index=False))

       model  uspešno  neuspelo  tokena ulaz  tokena izlaz  cena pilota $  cena 15.306 $
 gpt-oss-20b      194         6         1445           201          0.033           2.58
gpt-oss-120b      200         0         1445           236          0.072           5.49


### Slaganje modela

In [14]:
def jaccard(a: list[str], b: list[str]) -> float:
    A = {t.strip().upper() for t in a}
    B = {t.strip().upper() for t in b}
    if not A and not B:
        return 1.0
    return len(A & B) / len(A | B)


def cohen_kappa(x: pd.Series, y: pd.Series) -> float:
    """Slaganje korigovano za ono koje bi nastalo slučajno."""
    po = (x == y).mean()
    pe = sum((x == k).mean() * (y == k).mean() for k in set(x) | set(y))
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


par = rez_jeftin.merge(rez_skup, on="job_id", suffixes=("_j", "_s"))
print(f"Oglasa koje su oba modela obradila: {len(par)}\n")

par["jaccard"] = [jaccard(a, b) for a, b in zip(par["tech_stack_j"], par["tech_stack_s"])]

print(f"tech_stack — prosečan Jaccard : {par['jaccard'].mean():.3f}")
print(f"tech_stack — medijana Jaccard : {par['jaccard'].median():.3f}")
print(f"tech_stack — potpuno isto     : {(par['jaccard'] == 1).mean()*100:.1f}%")
print(f"tech_stack — prosečno tehnologija: {par['tech_stack_j'].apply(len).mean():.1f} (20b) "
      f"vs {par['tech_stack_s'].apply(len).mean():.1f} (120b)")

print(f"\nis_it_role  — slaganje : {(par['is_it_role_j'] == par['is_it_role_s']).mean()*100:.1f}%"
      f"   κ = {cohen_kappa(par['is_it_role_j'], par['is_it_role_s']):.3f}")
print(f"role_family — slaganje : {(par['role_family_j'] == par['role_family_s']).mean()*100:.1f}%"
      f"   κ = {cohen_kappa(par['role_family_j'], par['role_family_s']):.3f}")
print(f"vibe_score  — prosečna razlika : {(par['vibe_score_j'] - par['vibe_score_s']).abs().mean():.2f}")
print(f"red_flags   — prosečan broj    : {par['red_flags_j'].apply(len).mean():.2f} vs "
      f"{par['red_flags_s'].apply(len).mean():.2f}")
print(f"green_flags — prosečan broj    : {par['green_flags_j'].apply(len).mean():.2f} vs "
      f"{par['green_flags_s'].apply(len).mean():.2f}")

Oglasa koje su oba modela obradila: 194

tech_stack — prosečan Jaccard : 0.769
tech_stack — medijana Jaccard : 1.000
tech_stack — potpuno isto     : 51.5%
tech_stack — prosečno tehnologija: 4.4 (20b) vs 4.7 (120b)

is_it_role  — slaganje : 95.9%   κ = 0.883
role_family — slaganje : 86.1%   κ = 0.828
vibe_score  — prosečna razlika : 0.94
red_flags   — prosečan broj    : 0.19 vs 0.23
green_flags — prosečan broj    : 0.90 vs 0.87


### Gde se najviše ne slažu

In [15]:
neslaganje = par.nsmallest(8, "jaccard")[["title_j", "jaccard", "tech_stack_j", "tech_stack_s"]] \
    if "title_j" in par.columns else par.nsmallest(8, "jaccard")[["job_id", "jaccard", "tech_stack_j", "tech_stack_s"]]

for _, r in neslaganje.iterrows():
    print(f"\nJ={r['jaccard']:.2f}  job_id={r['job_id'] if 'job_id' in r else ''}")
    print(f"   20b : {r['tech_stack_j']}")
    print(f"   120b: {r['tech_stack_s']}")

print("\n\nOglasi gde se ne slažu oko is_it_role:")
sporni = par[par["is_it_role_j"] != par["is_it_role_s"]]
if len(sporni):
    naslovi = pilot.set_index("job_id")["title"]
    for _, r in sporni.head(10).iterrows():
        print(f"   {naslovi.get(r['job_id'], '?')[:60]:62} "
              f"20b={r['is_it_role_j']!s:5} 120b={r['is_it_role_s']!s:5}")
else:
    print("   nema — potpuno slaganje")


J=0.00  job_id=3802602037
   20b : ['iOS', 'Android']
   120b: []

J=0.00  job_id=3884856274
   20b : ['automation frameworks', 'load testing tools', 'cloud-based testing platforms']
   120b: []

J=0.00  job_id=3887719351
   20b : ['Microsoft Word', 'Microsoft Excel', 'Microsoft PowerPoint']
   120b: ['Excel', 'Word', 'PowerPoint']

J=0.00  job_id=3891275513
   20b : []
   120b: ['Microsoft OS', 'Server']

J=0.00  job_id=3898159427
   20b : ['Mobile', 'AI', 'Analytics']
   120b: []

J=0.00  job_id=3899524404
   20b : ['Linux RHEL', 'Blade Servers', 'Dell Servers', 'HP Blades', 'Storage']
   120b: ['Linux', 'RHEL']

J=0.00  job_id=3899528944
   20b : []
   120b: ['Linux', 'Windows', 'Cisco']

J=0.00  job_id=3900079723
   20b : []
   120b: ['DOORS', 'MS Excel', 'MS Word', 'MS Access']


Oglasi gde se ne slažu oko is_it_role:
   Danish, Norwegian - Linguistic Tester - Cupertino, CA          20b=True  120b=False
   ELECTRONICS DESIGN ENGINEER                                    20b=True  1

## 8. Uticaj dužine opisa

Prva verzija rada je opis sekla na 2.500 znakova, da bi se smanjio broj tokena
po zahtevu, a time i cena obrade. U pilotu 28 % oglasa nema
nijednu tehnologiju, a među njima su i tehničke pozicije:

```
Software Engineer II (Automation)     5.441 znakova
Senior Data Scientist                 6.275
Software Engineer - Fraud             8.811
R&D Scientific Programmer             7.008
```

Prva sumnja je da su ti oglasi duži, pa im je deo sa zahtevima odsečen:

| grupa | medijana dužine opisa | dužih od 2.500 |
|---|---:|---:|
| prazan tech_stack (n=54) | 5.849 | 94 % |
| nađene tehnologije (n=140) | 2.536 | 51 % |

Korelacija nije dokaz, jer duži oglasi mogu biti korporativni i bez konkretnih
alata. Zato se radi eksperiment sa kontrolnom grupom:

| grupa | šta radi | čemu služi |
|---|---|---|
| A | prazni oglasi, ponovljeni na 2.500 znakova | koliko se rezultat menja od samog ponavljanja |
| B | isti oglasi, pun opis | efekat ponavljanja i dužine zajedno |
| C | oglasi koji su imali tehnologije, pun opis | da li duži tekst nešto kvari |

Efekat skraćivanja je razlika između B i A.

### Rezultat

| grupa | prosek tehnologija | praznih |
|---|---:|---:|
| A, prazni na 2.500 znakova | 0,72 | 87 % |
| B, prazni na punom opisu | 3,59 | 54 % |
| C, neprazni na punom opisu | 7,37 | 13 % |
| isti oglasi ranije, na 2.500 znakova | 5,55 | |

Kontrolna grupa pokazuje da samo ponavljanje ne pomaže, jer A ostaje na 87 %
praznih. U grupi B broj tehnologija raste sa 0,72 na 3,59. Grupa C pokazuje da
su i oglasi sa nađenim tehnologijama gubili deo njih, 5,55 prema 7,37.

Primeri iz grupe B, svi ranije prazni:

```
Senior Software Engineer, Power Apps   C#, Java, JavaScript, Python, React, .NET, Azure
Information Security Advisor           AWS, Azure, CrowdStrike, Netskope, Splunk
Advanced Technology Grid AI/ML Lead    Python, C#, C/C++, Java, R, Kafka, AWS
```

Zbog toga konačna ekstrakcija koristi pun opis. Cena eksperimenta je 0,02 $.

In [16]:
# Merenje je već izvršeno; rezultati su u tabeli iznad.
# Postavi na True da ga ponoviš (~$0.02).
PONOVI_TEST_SKRACIVANJA = False

if PONOVI_TEST_SKRACIVANJA:
    p20  = ekstrahuj(pilot, MODEL_JEFTIN, oznaka="pilot", samo_kes=True)
    p120 = ekstrahuj(pilot, MODEL_SKUP,   oznaka="pilot", samo_kes=True)
    upareni = p20.merge(p120, on="job_id", suffixes=("_j", "_s"))
    oba_prazna = ((upareni["tech_stack_j"].apply(len) == 0)
                  & (upareni["tech_stack_s"].apply(len) == 0))

    prazni   = upareni[oba_prazna].sample(40, random_state=5)["job_id"]
    neprazni = upareni[~oba_prazna].sample(40, random_state=5)["job_id"]

    def _meri(job_ids, granica, ime):
        red = pilot[pilot["job_id"].isin(job_ids)]
        with ThreadPoolExecutor(8) as pool:
            rez = list(pool.map(
                lambda r: _obradi_jedan(r, MODEL_JEFTIN, granica),
                red.itertuples()))
        n = pd.Series([len(r["tech_stack"]) for r in rez if r])
        print(f"{ime:34} prosek={n.mean():.2f}  praznih={(n == 0).mean()*100:.0f}%")

    _meri(prazni,   2_500, "A  prazni @2500 (kontrola)")
    _meri(prazni,  20_000, "B  prazni @pun opis")
    _meri(neprazni, 20_000, "C  neprazni @pun opis")
else:
    print("Test se ne pokreće — rezultati su u markdown ćeliji iznad.")

Test se ne pokreće — rezultati su u markdown ćeliji iznad.


## 9. Ekstrakcija nad celim skupom

Izabran je `openai/gpt-oss-20b`. Slaganje sa `gpt-oss-120b` je 0,88 po Cohenovoj
kapi na `is_it_role` i 0,77 po Jaccardu na spisku tehnologija, uz upola nižu
cenu.

Ako se pokretanje prekine, sledeće nastavlja odakle je stalo.

### Probni run na 300 oglasa

Merenja u sekcijama 7 i 8 rađena su na skraćenim opisima. Pun opis menja broj
ulaznih tokena, cenu i stopu grešaka, pa se to proverava na 300 oglasa pre celog
skupa. Probni uzorak ide u isti keš, pa se ne plaća dvaput.

In [ ]:
MODEL_IZABRAN = "openai/gpt-oss-20b"

proba = filtrirani.sample(300, random_state=7)
rez_proba = ekstrahuj(proba, MODEL_IZABRAN)   

openai/gpt-oss-20b: sve 15,310 već u kešu


In [18]:
c_in, c_out = 0.075, 0.30      # $/1M tokena, Groq, gpt-oss-20b

ul = rez_proba["_tokena_ulaz"].mean()
iz = rez_proba["_tokena_izlaz"].mean()
broj_tech = rez_proba["tech_stack"].apply(len)

print(f"Obrađeno            : {len(rez_proba)} / {len(proba)}")
print(f"Tokena po oglasu    : {ul:.0f} ulaz / {iz:.0f} izlaz")
print(f"Tehnologija po oglasu: {broj_tech.mean():.2f}")
print(f"Praznih             : {(broj_tech == 0).mean()*100:.1f}%")
print(f"is_it_role = False  : {(~rez_proba['is_it_role']).sum()} "
      f"({(~rez_proba['is_it_role']).mean()*100:.1f}%)")

# Napomena: ovo je drugi uzorak oglasa nego u sekcijama 8 i 10, pa se brojevi
# ne smeju čitati kao uparena komparacija sa vrednostima na 2.500 znakova.
# Stroga, uparena komparacija je u sekciji 8.

cena_po_oglasu = (ul * c_in + iz * c_out) / 1e6
print(f"\nCena po oglasu      : ${cena_po_oglasu:.5f}")
print(f"Cena probe          : ${len(rez_proba) * cena_po_oglasu:.3f}")
print(f"Procena punog runa  : ${len(filtrirani) * cena_po_oglasu:.2f}  "
      f"({len(filtrirani):,} oglasa, {len(filtrirani) - len(rez_proba):,} preostalo)")

print("\nRaspodela role_family:")
print(rez_proba["role_family"].value_counts().to_string())

# --- Ko su oglasi bez ijedne tehnologije ---
print("\nPrazan tech_stack po is_it_role:")
print(pd.crosstab(rez_proba["is_it_role"], broj_tech == 0,
                  rownames=["is_it_role"], colnames=["prazan"]).to_string())

samo_it = rez_proba[rez_proba["is_it_role"]]
n_it = samo_it["tech_stack"].apply(len)
print(f"\nMeđu IT oglasima ({len(samo_it)}): praznih {(n_it == 0).sum()} "
      f"({(n_it == 0).mean()*100:.1f}%), prosek {n_it.mean():.2f} tehnologija")
print("Za poređenje: u prethodnoj verziji je 52,9% oglasa imalo nula klastera.")

# --- Projekcija na ceo skup ---
udeo_it = rez_proba["is_it_role"].mean()
n_sa_platom = 3_976                      # iz levka u sekciji 4
print(f"\nPROJEKCIJA NA CEO SKUP")
print(f"   filtriranih                    {len(filtrirani):>7,}")
print(f"   × is_it_role ({udeo_it*100:.0f}%)            ≈ {len(filtrirani)*udeo_it:>7,.0f} IT oglasa")
print(f"   sa validnom godišnjom platom   ≈ {n_sa_platom*udeo_it:>7,.0f}")
print(f"   od toga sa bar 1 tehnologijom  ≈ {n_sa_platom*udeo_it*(1-(n_it==0).mean()):>7,.0f}")
print(f"\n   prethodna verzija: 478 sa platom, od toga 225 sa tehnologijama")

Obrađeno            : 300 / 300
Tokena po oglasu    : 1836 ulaz / 199 izlaz
Tehnologija po oglasu: 6.40
Praznih             : 17.7%
is_it_role = False  : 47 (15.7%)

Cena po oglasu      : $0.00020
Cena probe          : $0.059
Procena punog runa  : $3.02  (15,306 oglasa, 15,006 preostalo)

Raspodela role_family:
role_family
software_engineering    105
data                     66
non_it                   49
it_support               29
devops_cloud             26
security                 20
qa                        4
management                1

Prazan tech_stack po is_it_role:
prazan      False  True 
is_it_role              
False          24     23
True          223     30

Među IT oglasima (253): praznih 30 (11.9%), prosek 7.17 tehnologija
Za poređenje: u prethodnoj verziji je 52,9% oglasa imalo nula klastera.

PROJEKCIJA NA CEO SKUP
   filtriranih                     15,306
   × is_it_role (84%)            ≈  12,908 IT oglasa
   sa validnom godišnjom platom   ≈   3,353
   od toga sa

In [19]:
MODEL_IZABRAN = "openai/gpt-oss-20b"

# Ceo skup: 15.306 oglasa, oko 3 $, oko 20 minuta.
POKRENI_PUN_RUN = True

rezultat = ekstrahuj(
    filtrirani,
    MODEL_IZABRAN,
    niti=8,
    samo_kes=not POKRENI_PUN_RUN,
)

if not POKRENI_PUN_RUN:
    print("\nPun run nije pokrenut. Postavi POKRENI_PUN_RUN = True pa pokreni ćeliju.")
else:
    print(f"\nObrađeno: {len(rezultat):,} / {len(filtrirani):,}")
    print(f"Potrošeno tokena: {rezultat['_tokena_ulaz'].sum():,} ulaz / "
          f"{rezultat['_tokena_izlaz'].sum():,} izlaz")

openai/gpt-oss-20b: sve 15,310 već u kešu



Obrađeno: 15,306 / 15,306
Potrošeno tokena: 27,572,258 ulaz / 3,140,331 izlaz


Po oglasu to je 27.572.258 / 15.306 = 1.801 ulazni i 3.140.331 / 15.306 = 205
izlaznih tokena. Po cenama iz sekcije 7 (0,075 $ i 0,30 $ za milion tokena)
cela obrada košta 27,57 · 0,075 + 3,14 · 0,30 = 3,01 $. Procena iz sekcije 7,
2,58 $, bila je niža jer je rađena na opisima skraćenim na 2.500 znakova, sa
1.445 ulaznih tokena po oglasu.

## 10. Poređenje sa modelom iz prethodne verzije

Prethodna verzija projekta koristila je `llama-3.3-70b-versatile`. Model je
promenjen iz dva razloga.

Prvi je tehnički. Taj model na Groq-u ne podržava strogu proveru šeme, pa JSON
piše kao slobodan tekst. Kada odstupi, ceo poziv pada, i tako je u prethodnoj
verziji izgubljeno 40 od 1.500 oglasa.

Drugi je kvalitet izdvojenih pojmova. Isti skup od 895 oglasa je provučen kroz
oba modela, sa istim skraćivanjem na 2.500 znakova:

| | llama-3.3-70b | gpt-oss-20b |
|---|---:|---:|
| tehnologija po oglasu | 5,35 | 5,18 |
| oglasa bez tehnologija | 8,5 % | 22,1 % |
| udeo pojmova koji nisu tehnologije | 4,0 % | 2,1 % |

Llama nalazi više pojmova, ali su među njima UPS UNITS, GENERATORS, CABLING i
HVAC SYSTEMS. Za mrežu tehnologija je preciznost važnija od odziva, jer pogrešan
pojam stvara i pogrešan klaster. Veći udeo praznih oglasa kod gpt-oss-a dolazi
od skraćivanja opisa i nestaje sa punim opisom, kako pokazuje sekcija 8.

In [20]:
import ast

# Ekstrakcija za ovo poređenje je već urađena i keširana (data/interim/ekstrakcija_vs-llama_*).
# Postavi na True samo ako je želiš ponoviti (~$0.17).
PONOVI_POREDJENJE_SA_LLAMOM = False

_lista = lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])

stara = pd.read_csv(INTERIM / "stara_ekstrakcija_llama33.csv")
for kol in ["tech_stack", "red_flags", "green_flags"]:
    stara[kol] = stara[kol].apply(_lista)

zajednicki = filtrirani[filtrirani["job_id"].isin(stara["job_id"])]
print(f"Stara ekstrakcija: {len(stara):,} oglasa | prolazi novi filter: {len(zajednicki):,}\n")

rez_oss = ekstrahuj(
    zajednicki, MODEL_JEFTIN,
    oznaka="vs-llama",
    granica=2500,                          # isto skraćivanje kao u staroj verziji
    samo_kes=not PONOVI_POREDJENJE_SA_LLAMOM,
)

uporedo = stara[["job_id", "tech_stack", "vibe_score", "red_flags", "green_flags"]].merge(
    rez_oss[["job_id", "tech_stack", "vibe_score", "red_flags", "green_flags"]],
    on="job_id", suffixes=("_lla", "_oss"))
print(f"Uparenih oglasa: {len(uporedo):,}")

Stara ekstrakcija: 1,460 oglasa | prolazi novi filter: 1,031

openai/gpt-oss-20b: iz keša 895 | neobrađeno 136 (API nije pozvan)
Uparenih oglasa: 895


In [21]:
# Pojmovi koji su ekstrahovani kao "tehnologija", a to nisu.
NIJE_TEHNOLOGIJA = {
    "UPS UNITS", "GENERATORS", "CABLING", "HVAC SYSTEMS", "HVAC", "ELECTRICAL SYSTEMS",
    "CAN-BASED COMMUNICATION", "OSCILLOSCOPES", "MULTIMETERS", "PNEUMATIC CONTROLS",
    "MOTOR CONTROLLER", "EQUIPMENT", "REPAIRS", "PLUMBING", "WELDING", "OSHA",
    "AUTOMOTIVE", "MECHANICAL", "LIFTING", "DRIVING", "CALIBRATIONS",
    "PREVENTATIVE MAINTENANCE", "COMPUTER SCIENCE", "SOFTWARE ENGINEERING",
    "INFORMATION TECHNOLOGY", "COMPUTER ENGINEERING", "DEGREE", "CIVIL ENGINEERING",
    "MICROSOFT OFFICE", "MICROSOFT OFFICE SUITE", "EXCEL", "WORD", "POWERPOINT",
    "OUTLOOK", "TEAMS", "VISIO", "SHAREPOINT", "ACCESS", "DATA ENTRY", "TYPING",
    "AGILE", "SCRUM", "COMMUNICATION", "TEAMWORK", "PROBLEM SOLVING", "MEETINGS",
    "PLANNING", "WRITING", "DESIGN", "TESTING", "DEBUGGING", "TROUBLESHOOTING",
    "MAINTENANCE", "DOCUMENTATION", "CODE REVIEW", "MENTORING", "BEST PRACTICES",
    "NETWORKING", "PROTOCOLS", "CATIA", "SOLIDWORKS", "AUTOCAD", "LABVIEW", "PLC", "MATLAB",
}

_skup = lambda lista: {str(t).strip().upper() for t in lista}

uporedo["jaccard"] = [
    len(_skup(a) & _skup(b)) / max(len(_skup(a) | _skup(b)), 1) if (a or b) else 1.0
    for a, b in zip(uporedo["tech_stack_lla"], uporedo["tech_stack_oss"])
]

redovi = []
for ime, kol in [("llama-3.3-70b", "tech_stack_lla"), ("gpt-oss-20b", "tech_stack_oss")]:
    broj = uporedo[kol].apply(len)
    sve = [str(t).strip().upper() for lista in uporedo[kol] for t in lista]
    redovi.append({
        "model": ime,
        "tehnologija/oglas": round(broj.mean(), 2),
        "praznih %": round((broj == 0).mean() * 100, 1),
        "ukupno pojava": len(sve),
        "nije tehnologija %": round(sum(t in NIJE_TEHNOLOGIJA for t in sve) / len(sve) * 100, 1),
    })
print(pd.DataFrame(redovi).to_string(index=False))

print(f"\nJaccard tech_stack : prosek {uporedo['jaccard'].mean():.3f} · "
      f"medijana {uporedo['jaccard'].median():.3f} · "
      f"identično {(uporedo['jaccard'] == 1).mean()*100:.1f}%")
for kol, ime in [("red_flags", "red_flags"), ("green_flags", "green_flags")]:
    print(f"{ime:12} po oglasu : {uporedo[kol+'_lla'].apply(len).mean():.2f} (llama) "
          f"vs {uporedo[kol+'_oss'].apply(len).mean():.2f} (gpt-oss)")
print(f"vibe_score   prosek   : {uporedo['vibe_score_lla'].mean():.2f} (llama) "
      f"vs {uporedo['vibe_score_oss'].mean():.2f} (gpt-oss)")

# Kvalitet onoga sto svaki model nadje a drugi ne
c_lla = Counter(t for lista in uporedo["tech_stack_lla"] for t in _skup(lista))
c_oss = Counter(t for lista in uporedo["tech_stack_oss"] for t in _skup(lista))

print("\nKVALITET RAZLIKE — šta jedan model nađe a drugi ne:")
for razlika, ime in [(c_lla - c_oss, "samo llama-3.3-70b"), (c_oss - c_lla, "samo gpt-oss-20b")]:
    ukupno = sum(razlika.values())
    lose = sum(v for k, v in razlika.items() if k in NIJE_TEHNOLOGIJA)
    print(f"  {ime:22} {ukupno:5} pojava, {lose:4} nisu tehnologije ({lose/ukupno*100:.1f}%)")
    print(f"  {'':22} {[t for t, _ in razlika.most_common(10)]}")

        model  tehnologija/oglas  praznih %  ukupno pojava  nije tehnologija %
llama-3.3-70b               5.35        8.5           4789                 4.0
  gpt-oss-20b               5.18       22.1           4632                 2.1

Jaccard tech_stack : prosek 0.582 · medijana 0.667 · identično 27.8%
red_flags    po oglasu : 0.36 (llama) vs 0.30 (gpt-oss)
green_flags  po oglasu : 1.06 (llama) vs 0.70 (gpt-oss)
vibe_score   prosek   : 6.63 (llama) vs 5.37 (gpt-oss)

KVALITET RAZLIKE — šta jedan model nađe a drugi ne:
  samo llama-3.3-70b      1071 pojava,   98 nisu tehnologije (9.2%)
                         ['AI', 'MACHINE LEARNING', 'WORD', 'UPS UNITS', 'MICROSOFT OFFICE SUITE', 'AGILE', 'CABLING', 'GENERATORS', 'CISCO', 'AMAZON WEB SERVICES']
  samo gpt-oss-20b         915 pojava,    4 nisu tehnologije (0.4%)
                         ['CI/CD', 'SQL', 'REST', 'CLOUD', 'PYTHON', 'KUBERNETES', 'AWS', 'DOCKER', 'GIT', 'MICROSERVICES']


## 11. Sklapanje skupa

Spajaju se rezultat ekstrakcije, kolone iz `postings.csv` i plate iz
`salaries.csv`.

### Plate

Uzima se sredina raspona, `(min + max) / 2`, a `med_salary` kada raspon
nedostaje. Svi podaci o plati dolaze iz iste tabele, i zadržavaju se samo iznosi
u dolarima.

Godišnji faktori: HOURLY × 2080 (40 sati puta 52 nedelje), WEEKLY × 52,
BIWEEKLY × 26, MONTHLY × 12, YEARLY × 1.

Opseg od 20.000 do 500.000 dolara primenjuje se pri analizi, isto kao u
`04_plate.ipynb`.

### Duplikati

Isti oglas se javlja pod više job_id vrednosti: ista firma, isti naslov, isti
opis. DataAnnotation je jedan oglas objavio 39 puta. Ako takvi oglasi ostanu,
isti par tehnologija se broji više puta, a ista opažanja ulaze u korelacije kao
nezavisna. Zadržava se prvo pojavljivanje svake kombinacije firme, naslova i
opisa.

In [22]:
ekstrakcija = ekstrahuj(filtrirani, MODEL_IZABRAN, samo_kes=True)
print(f"Ekstrahovano: {len(ekstrakcija):,} / {len(filtrirani):,}")

# --- plate ---
FAKTOR = {"YEARLY": 1, "MONTHLY": 12, "WEEKLY": 52, "BIWEEKLY": 26, "HOURLY": 2080}

plate = salaries[salaries["currency"] == "USD"].copy()
plate["sredina"] = plate[["min_salary", "max_salary"]].mean(axis=1).fillna(plate["med_salary"])
plate["godisnja_plata"] = plate["sredina"] * plate["pay_period"].map(FAKTOR)

# --- spajanje ---
dataset = (
    ekstrakcija
    .drop(columns=["reasoning", "_tokena_ulaz", "_tokena_izlaz"])
    .merge(postings[["job_id", "title", "company_name", "location",
                     "views", "applies", "formatted_experience_level",
                     "remote_allowed", "formatted_work_type"]], on="job_id", how="left")
    .merge(plate[["job_id", "godisnja_plata", "pay_period"]], on="job_id", how="left")
)

assert dataset["job_id"].is_unique, "duplirani job_id nakon spajanja"
assert len(dataset) == len(ekstrakcija), "spajanje je promenilo broj redova"

# --- duplikati: isti oglas pod više job_id ---
import hashlib
opis = postings.set_index("job_id")["description"].astype(str).str.strip().str.lower()
dataset["_opis_hash"] = dataset["job_id"].map(opis).apply(
    lambda s: hashlib.md5(s.encode()).hexdigest())
kljuc = ["company_name", "title", "_opis_hash"]
pre_dedup = len(dataset)
dataset = (dataset.sort_values("job_id")
                  .drop_duplicates(subset=kljuc, keep="first")
                  .drop(columns="_opis_hash")
                  .reset_index(drop=True))
print(f"Duplikati uklonjeni: {pre_dedup:,} → {len(dataset):,}  "
      f"({pre_dedup - len(dataset):,} identičnih kopija)")

dataset.to_parquet(INTERIM / "dataset.parquet", index=False)
print(f"Upisano: data/interim/dataset.parquet  ({len(dataset):,} redova, "
      f"{len(dataset.columns)} kolona)")

openai/gpt-oss-20b: iz keša 15,310 | neobrađeno 0 (API nije pozvan)
Ekstrahovano: 15,306 / 15,306


Duplikati uklonjeni: 15,306 → 14,156  (1,150 identičnih kopija)
Upisano: data/interim/dataset.parquet  (14,156 redova, 17 kolona)


### Finalni brojevi

In [23]:
it = dataset[dataset["is_it_role"]]
u_opsegu = it["godisnja_plata"].between(20_000, 500_000)
ima_tech = it["tech_stack"].apply(len) > 0

redovi = [
    ("filtrirano (slojevi 1+2)",        len(filtrirani)),
    ("ekstrahovano",                    len(dataset)),
    ("is_it_role = True (sloj 3)",      len(it)),
    ("+ ima platu u opsegu 20k–500k",   int(u_opsegu.sum())),
    ("+ ima bar 1 tehnologiju",         int((u_opsegu & ima_tech).sum())),
]
for naziv, n in redovi:
    print(f"   {naziv:34} {n:>7,}")

print(f"\n   tehnologija po IT oglasu       {it['tech_stack'].apply(len).mean():>7.2f}")
print(f"   IT oglasa bez tehnologija      {(it['tech_stack'].apply(len) == 0).mean()*100:>6.1f}%")
print(f"   medijana godišnje plate        {it.loc[u_opsegu, 'godisnja_plata'].median():>7,.0f} $")

print("\nRaspodela role_family:")
print(dataset["role_family"].value_counts().to_string())

   filtrirano (slojevi 1+2)            15,306
   ekstrahovano                        14,156
   is_it_role = True (sloj 3)          11,713
   + ima platu u opsegu 20k–500k        3,411
   + ima bar 1 tehnologiju              3,073

   tehnologija po IT oglasu          7.34
   IT oglasa bez tehnologija         8.9%
   medijana godišnje plate        131,600 $

Raspodela role_family:
role_family
software_engineering    4513
data                    2914
non_it                  2511
devops_cloud            1557
it_support              1328
security                1022
qa                       229
management                82


## 12. Ručna provera filtera

Slaganje dva modela pokazuje da su odgovori stabilni, ali ne i da su tačni, jer
modeli dele poreklo i greške. Zato je uzorak od 50 oglasa iz konačnog skupa
označen ručno, prema definiciji zapisanoj u `validacija/validacija_3_uputstvo.txt`.
Ocena modela je pri označavanju bila sakrivena.

Ćelija ispod računa precision i recall kada se kolona u `validacija/validacija_3.csv`
popuni.

In [ ]:
KRUG = 3
csv_uzorka = Path("validacija") / f"validacija_{KRUG}.csv"
kljuc_uzorka = INTERIM / f"validacija_{KRUG}_kljuc.json"

ocene = pd.read_csv(csv_uzorka, encoding="utf-8-sig")
kolona = "OCENA_da_ne" if "OCENA_da_ne" in ocene.columns else "TVOJA_OCENA_da_ne"
ocene["rucno"] = (ocene[kolona].astype(str).str.strip().str.lower()
                  .map({"da": True, "d": True, "ne": False, "n": False}))

if ocene["rucno"].isna().any():
    print(f"Uzorak još nije označen: prazno je {ocene['rucno'].isna().sum()} od {len(ocene)} redova.")
    print(f"Popuniti kolonu {kolona} u {csv_uzorka}, pa pokreni ćeliju ponovo.")
else:
    ocene["model"] = ocene["job_id"].astype(str).map(json.load(open(kljuc_uzorka)))
    TP = int((ocene["model"] & ocene["rucno"]).sum())
    FP = int((ocene["model"] & ~ocene["rucno"]).sum())
    FN = int((~ocene["model"] & ocene["rucno"]).sum())
    TN = int((~ocene["model"] & ~ocene["rucno"]).sum())
    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    print(f"uzorak: {len(ocene)} oglasa\n")
    print(f"{'':16}{'ručno IT':>10}{'ručno nije IT':>16}")
    print(f"{'model IT':16}{TP:>10}{FP:>16}")
    print(f"{'model nije IT':16}{FN:>10}{TN:>16}")
    print(f"\ntačnost   {(TP + TN) / len(ocene):.3f}")
    print(f"precision {precision:.3f}")
    print(f"recall    {recall:.3f}")
    print(f"F1        {f1:.3f}")

    # Wilsonov 95 % interval poverenja za preciznost, jer je uzorak mali
    from statsmodels.stats.proportion import proportion_confint
    donja, gornja = proportion_confint(TP, TP + FP, alpha=0.05, method="wilson")
    print(f"\n95 % interval za precision (Wilson): {donja:.2f} do {gornja:.2f}")

uzorak: 50 oglasa

                  ručno IT   ručno nije IT
model IT                37               4
model nije IT            0               9

tačnost   0.920
precision 0.902
recall    1.000
F1        0.949

95 % interval za precision (Wilson): 0.77 do 0.96
